# IBM Quantum Workshop

1. Set your options in the **SETUP** cell below, then run it.
2. Run the exercises in order.

The values at the top of each exercise can be changed and the cell re-run.

In [ ]:
# ============ SETUP - edit these, then run this cell ============

USE_SIMULATOR = True   # True = local simulator, False = your real IBM Quantum instance

# Only needed if USE_SIMULATOR = False:
API_KEY = ""           # 44-character API key from your IBM Quantum dashboard
CRN = ""               # starts with crn:v1:bluemix... (Instances page)


# --- nothing to change below this line ---
from workshop import setup

setup(USE_SIMULATOR, API_KEY, CRN)

The simulator needs no credentials.

For real hardware, set `USE_SIMULATOR = False` and paste your API key and CRN
between the quotes. Both are checked when the cell runs, so a bad key or CRN is
reported here rather than further down.

---
# Exercise 1 - Rotating a qubit

A classical bit is 0 or 1. A qubit carries a weighted chance of each and settles
on one only when measured.

The `ry` gate rotates the qubit. The angle sets the probability of measuring a `1`:

$$P(1) = \sin^2(\theta/2)$$

A quarter turn gives an even split. Larger angles make `1` more likely.

In [ ]:
# ============ EXERCISE 1 - a biased qubit ============

P_ONE = 0.50    # chance of measuring 1.  Try: 0.10, 0.25, 0.50, 0.75, 0.90

# ---------------------------------------------------------------
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from IPython.display import display

from workshop import run_circuit

# Angle giving P_ONE, from P(1) = sin^2(theta/2).
theta = 2 * np.arcsin(np.sqrt(P_ONE))

qubit = QuantumCircuit(QuantumRegister(1, 'q'), ClassicalRegister(1, 'meas'))
qubit.ry(theta, 0)
qubit.measure(0, 0)

print(f'theta = {theta:.3f} rad, P(1) = {P_ONE:.0%}')
display(qubit.draw('mpl'))

# The same circuit, run five times at one shot each.
print('\nFive separate measurements:')
for i in range(5):
    counts = run_circuit(qubit, shots=1)
    outcome = list(counts)[0]
    print(f'  measurement {i + 1}: {outcome}', flush=True)

Five measurements is a small sample. At `P_ONE = 0.5` all five can still come out
the same. Re-run the cell for a different five.

The cell below measures the same circuit 2000 times, where the bias is visible.

In [ ]:
# ============ EXERCISE 1b - the same qubit, 2000 measurements ============
from qiskit.visualization import plot_histogram

counts = run_circuit(qubit, shots=2000)
measured = counts.get('1', 0) / sum(counts.values())

print(f'requested P(1): {P_ONE:.0%}')
print(f'measured  P(1): {measured:.1%}   {counts}')

display(plot_histogram(counts))

---
# Exercise 2 - Entanglement

A Hadamard puts the first qubit into superposition. A CNOT then links the second
qubit to it.

Measuring gives `00` or `11`, never `01` or `10`. Each qubit on its own is an even
split, but the two always agree.

In [ ]:
# ============ EXERCISE 2 - a Bell pair ============
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.visualization import plot_histogram
from IPython.display import display

from workshop import run_circuit

bell = QuantumCircuit(QuantumRegister(2, 'q'), ClassicalRegister(2, 'meas'))
bell.h(0)                       # superposition
bell.cx(0, 1)                   # entangle
bell.measure([0, 1], [0, 1])

display(bell.draw('mpl'))

counts = run_circuit(bell, shots=1024)
print('Counts:', counts)
print('Outcomes seen:', sorted(counts))

display(plot_histogram(counts))

---
# Exercise 3 - Shor's algorithm

Factoring 15 by period finding. The quantum part finds the period of $a^x \bmod N$,
and the factors follow from a gcd.

This is the algorithm behind quantum attacks on RSA. Keys of a useful size are far
beyond current hardware.

In [ ]:
# ============ EXERCISE 3 - Shor's algorithm ============

N = 15          # the number to factor
a = 7           # try 2, 4, 7, 8, 11 or 13
N_COUNT = 3     # counting qubits

# ---------------------------------------------------------------
from fractions import Fraction
from math import gcd

import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from IPython.display import display

from workshop import run_circuit


def c_amod15(a, power):
    """Controlled multiplication by a^power mod 15.

    Hard-coded for 15: each valid `a` permutes the four work qubits, so the
    modular arithmetic reduces to a pattern of swaps.
    """
    U = QuantumCircuit(4)
    for _ in range(power):
        if a in [2, 13]:
            U.swap(2, 3); U.swap(1, 2); U.swap(0, 1)
        if a in [7, 8]:
            U.swap(0, 1); U.swap(1, 2); U.swap(2, 3)
        if a in [4, 11]:
            U.swap(1, 3); U.swap(0, 2)
        if a in [7, 11, 13]:
            for q in range(4):
                U.x(q)
    gate = U.to_gate()
    gate.name = f'{a}^{power} mod 15'
    return gate.control()


def iqft(n):
    """Inverse quantum Fourier transform.

    Converts the phases held in the counting register into a bit string that
    can be measured.
    """
    qc = QuantumCircuit(n)
    for q in range(n // 2):
        qc.swap(q, n - 1 - q)
    for j in range(n):
        for m in range(j):
            qc.cp(-np.pi / 2 ** (j - m), m, j)
        qc.h(j)
    gate = qc.to_gate()
    gate.name = 'IQFT'
    return gate


# Period-finding circuit.
shor = QuantumCircuit(
    QuantumRegister(N_COUNT, 'count'),
    QuantumRegister(4, 'work'),
    ClassicalRegister(N_COUNT, 'meas'),
)
for q in range(N_COUNT):
    shor.h(q)
shor.x(N_COUNT)                     # work register starts at |1>
for q in range(N_COUNT):
    shor.append(c_amod15(a, 2 ** q), [q] + list(range(N_COUNT, N_COUNT + 4)))
shor.append(iqft(N_COUNT), range(N_COUNT))
shor.measure(range(N_COUNT), range(N_COUNT))

display(shor.draw('mpl', fold=-1))

counts = run_circuit(shor, shots=1024)
print('Measured phases:', counts)

# Classical part: phase -> period -> factors.
factors = set()
for bits in sorted(counts, key=counts.get, reverse=True):
    phase = int(bits, 2) / 2 ** N_COUNT
    r = Fraction(phase).limit_denominator(N).denominator
    if r % 2 == 0:
        for f in (gcd(a ** (r // 2) - 1, N), gcd(a ** (r // 2) + 1, N)):
            if f not in (1, N):
                factors.add(f)

if factors:
    print(f'\nFactors of {N}: {sorted(factors)}')
else:
    print('\nNo non-trivial factor from this run. Re-run the cell.')